<a href="https://colab.research.google.com/github/CassieMarie0728/colab-notebooks/blob/main/firecrawl_site_crawl_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Firecrawl Site Crawl Notebook

This Colab notebook crawls a site with Firecrawl and requests multiple output formats, including structured JSON extraction.

**What changed from your pasted code:**
- Removed the hardcoded API key so you don't leave secrets lying around.
- Switched to the current Python SDK crawl pattern using `scrape_options`, which Firecrawl documents for recursive site crawls. citeturn933107search1turn933107search2turn933107search10
- Kept a detailed schema, but used a flatter version that is less likely to make validators throw a little hissy fit.

Edit the `START_URL` and `DOMAIN_LABEL` cell, then run top to bottom.

In [ ]:
# Install the Firecrawl Python SDK
!pip -q install firecrawl-py

In [ ]:
import os
import json
import zipfile
from pathlib import Path

from firecrawl import Firecrawl

try:
    from google.colab import userdata, files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    userdata = None
    files = None

## 1) Set your Firecrawl API key

Best move in Colab:
- Open the **key icon / Secrets** panel
- Add a secret named **`FIRECRAWL_API_KEY`**

You can also paste it directly below if you absolutely must, but secrets are the non-chaotic option.

In [ ]:
API_KEY = None

if IN_COLAB:
    try:
        API_KEY = userdata.get('FIRECRAWL_API_KEY')
    except Exception:
        API_KEY = None

if not API_KEY:
    API_KEY = os.environ.get('FIRECRAWL_API_KEY')

# Last-resort manual paste option:
API_KEY = 'fc-cd9xxxxxxxxxx'

if not API_KEY:
    raise ValueError('No Firecrawl API key found. Add FIRECRAWL_API_KEY in Colab Secrets or set it manually.')

app = Firecrawl(api_key=API_KEY)
print('Firecrawl client ready.')

## 2) Choose the site you want to crawl

In [ ]:
# Pick one
START_URL = 'http://nobullshitgrief.com'
DOMAIN_LABEL = 'nobullshitgrief.com'

# Example swap:
# START_URL = 'https://cassandracrossno.com'
# DOMAIN_LABEL = 'cassandracrossno.com'

# Crawl controls
LIMIT = 75              # raise or lower as needed
ONLY_MAIN_CONTENT = False
MAX_AGE_MS = 172800000   # 2 days cache
PARSERS = ['pdf']

## 3) Prompt for JSON extraction

This stays short on purpose so it doesn't smash into prompt-size nonsense.

In [ ]:
JSON_PROMPT = (
    'Extract structured JSON from this domain crawl. Classify each page by role, '
    'extract only explicit facts, do not invent, normalize repeated sitewide info, '
    'preserve exact titles, headlines, slogans, names, and CTAs, return concise '
    'summaries, identify page purpose and key entities, and separate sitewide brand '
    'data from page-specific content.'
)

## 4) Detailed JSON schema

This is the flatter version meant to be less likely to trigger validator drama.

In [ ]:
SCHEMA = {
    'type': 'object',
    'additionalProperties': False,
    'properties': {
        'site': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'domain': {'type': 'string'},
                'site_name': {'type': ['string', 'null']},
                'site_tagline': {'type': ['string', 'null']},
                'site_description': {'type': ['string', 'null']},
                'primary_brand_name': {'type': ['string', 'null']},
                'site_type': {'type': 'array', 'items': {'type': 'string'}},
                'primary_audience': {'type': 'array', 'items': {'type': 'string'}},
                'brand_voice': {'type': 'array', 'items': {'type': 'string'}},
                'core_topics': {'type': 'array', 'items': {'type': 'string'}},
                'mission_summary': {'type': ['string', 'null']},
                'primary_conversion_goals': {'type': 'array', 'items': {'type': 'string'}},
                'top_navigation': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'label': {'type': 'string'},
                            'url': {'type': 'string'}
                        },
                        'required': ['label', 'url']
                    }
                },
                'social_links': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'platform': {'type': 'string'},
                            'url': {'type': 'string'}
                        },
                        'required': ['platform', 'url']
                    }
                },
                'contact_points': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'type': {'type': 'string'},
                            'value': {'type': 'string'}
                        },
                        'required': ['type', 'value']
                    }
                }
            },
            'required': [
                'domain', 'site_name', 'site_tagline', 'site_description',
                'primary_brand_name', 'site_type', 'primary_audience', 'brand_voice',
                'core_topics', 'mission_summary', 'primary_conversion_goals',
                'top_navigation', 'social_links', 'contact_points'
            ]
        },
        'sitewide_entities': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'people': {'type': 'array', 'items': {'type': 'object'}},
                'books': {'type': 'array', 'items': {'type': 'object'}},
                'products': {'type': 'array', 'items': {'type': 'object'}},
                'organizations': {'type': 'array', 'items': {'type': 'object'}},
                'projects': {'type': 'array', 'items': {'type': 'object'}},
                'topics': {'type': 'array', 'items': {'type': 'object'}}
            },
            'required': ['people', 'books', 'products', 'organizations', 'projects', 'topics']
        },
        'content_inventory': {
            'type': 'object',
            'additionalProperties': False,
            'properties': {
                'total_pages_crawled': {'type': ['integer', 'null']},
                'page_type_counts': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'page_type': {'type': 'string'},
                            'count': {'type': 'integer'}
                        },
                        'required': ['page_type', 'count']
                    }
                },
                'important_pages': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'url': {'type': 'string'},
                            'reason': {'type': 'string'}
                        },
                        'required': ['url', 'reason']
                    }
                }
            },
            'required': ['total_pages_crawled', 'page_type_counts', 'important_pages']
        },
        'offerings': {
            'type': 'array',
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'name': {'type': 'string'},
                    'offering_type': {'type': 'string'},
                    'description': {'type': ['string', 'null']},
                    'price_text': {'type': ['string', 'null']},
                    'url': {'type': ['string', 'null']},
                    'cta': {'type': ['string', 'null']}
                },
                'required': ['name', 'offering_type', 'description', 'price_text', 'url', 'cta']
            }
        },
        'pages': {
            'type': 'array',
            'items': {
                'type': 'object',
                'additionalProperties': False,
                'properties': {
                    'url': {'type': 'string'},
                    'canonical_url': {'type': ['string', 'null']},
                    'slug': {'type': ['string', 'null']},
                    'page_type': {'type': 'string'},
                    'title': {'type': ['string', 'null']},
                    'meta_title': {'type': ['string', 'null']},
                    'meta_description': {'type': ['string', 'null']},
                    'h1': {'type': ['string', 'null']},
                    'published_date': {'type': ['string', 'null']},
                    'modified_date': {'type': ['string', 'null']},
                    'author_name': {'type': ['string', 'null']},
                    'summary': {'type': 'string'},
                    'purpose': {'type': 'array', 'items': {'type': 'string'}},
                    'tone': {'type': 'array', 'items': {'type': 'string'}},
                    'topics': {'type': 'array', 'items': {'type': 'string'}},
                    'taxonomy': {
                        'type': 'object',
                        'additionalProperties': False,
                        'properties': {
                            'categories': {'type': 'array', 'items': {'type': 'string'}},
                            'tags': {'type': 'array', 'items': {'type': 'string'}}
                        },
                        'required': ['categories', 'tags']
                    },
                    'key_points': {'type': 'array', 'items': {'type': 'string'}},
                    'important_quotes': {'type': 'array', 'items': {'type': 'string'}},
                    'primary_ctas': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'additionalProperties': False,
                            'properties': {
                                'label': {'type': 'string'},
                                'destination_url': {'type': ['string', 'null']},
                                'cta_type': {'type': 'string'}
                            },
                            'required': ['label', 'destination_url', 'cta_type']
                        }
                    },
                    'internal_links': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'additionalProperties': False,
                            'properties': {
                                'url': {'type': 'string'},
                                'label': {'type': ['string', 'null']},
                                'link_type': {'type': 'string'}
                            },
                            'required': ['url', 'label', 'link_type']
                        }
                    },
                    'external_links': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'additionalProperties': False,
                            'properties': {
                                'url': {'type': 'string'},
                                'label': {'type': ['string', 'null']},
                                'link_type': {'type': 'string'}
                            },
                            'required': ['url', 'label', 'link_type']
                        }
                    },
                    'media_assets': {
                        'type': 'array',
                        'items': {
                            'type': 'object',
                            'additionalProperties': False,
                            'properties': {
                                'asset_type': {'type': 'string'},
                                'url': {'type': ['string', 'null']},
                                'alt_or_label': {'type': ['string', 'null']}
                            },
                            'required': ['asset_type', 'url', 'alt_or_label']
                        }
                    },
                    'relationship_hints': {'type': 'array', 'items': {'type': 'string'}}
                },
                'required': [
                    'url', 'canonical_url', 'slug', 'page_type', 'title', 'meta_title',
                    'meta_description', 'h1', 'published_date', 'modified_date',
                    'author_name', 'summary', 'purpose', 'tone', 'topics', 'taxonomy',
                    'key_points', 'important_quotes', 'primary_ctas', 'internal_links',
                    'external_links', 'media_assets', 'relationship_hints'
                ]
            }
        }
    },
    'required': ['site', 'content_inventory', 'offerings', 'pages']
}

## 5) Build crawl formats

Firecrawl supports JSON extraction by passing an object inside `formats`, and crawl accepts scrape options that are applied to each page. citeturn933107search0turn933107search2turn933107search5turn933107search10

In [ ]:
FORMATS = [
    'markdown',
    'summary',
    'links',
    'html',
    'branding',
    'images',
    {'type': 'screenshot', 'fullPage': True},
    {
        'type': 'json',
        'prompt': JSON_PROMPT,
        'schema': SCHEMA,
    },
]

SCRAPE_OPTIONS = {
    'only_main_content': ONLY_MAIN_CONTENT,
    'max_age': MAX_AGE_MS,
    'parsers': PARSERS,
    'formats': FORMATS,
}

print('Formats ready:', len(FORMATS))

## 6) Run the crawl

This is the part where you let the gremlin off the leash.

In [ ]:
crawl_result = app.crawl(
    START_URL,
    limit=LIMIT,
    scrape_options=SCRAPE_OPTIONS,
)

type(crawl_result), crawl_result

## 7) Save the results

This writes the raw crawl response plus a ZIP copy you can download from Colab.

In [ ]:
output_dir = Path('/content/firecrawl_output')
output_dir.mkdir(parents=True, exist_ok=True)

json_path = output_dir / f'{DOMAIN_LABEL}-crawl-result.json'
zip_path = output_dir / f'{DOMAIN_LABEL}-crawl-result.zip'

with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(crawl_result, f, indent=2, ensure_ascii=False, default=str)

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(json_path, arcname=json_path.name)

print('Saved JSON to:', json_path)
print('Saved ZIP to:', zip_path)

## 8) Optional: quick preview

In [ ]:
if isinstance(crawl_result, dict):
    print('Top-level keys:', list(crawl_result.keys())[:20])
    if 'data' in crawl_result and isinstance(crawl_result['data'], list):
        print('Pages returned:', len(crawl_result['data']))
        if crawl_result['data']:
            first = crawl_result['data'][0]
            print('First page keys:', list(first.keys())[:30])
else:
    print('Crawl result type:', type(crawl_result))

## 9) Download the result from Colab

In [ ]:
if IN_COLAB:
    files.download(str(zip_path))
else:
    print('Not running in Colab. Download manually from:', zip_path)